In [ ]:
import os
import math
import random
import numpy as np
import torch

import rasterio
from rasterio.windows import Window

from torch.utils.data import Dataset, DataLoader
from tqdm.auto import tqdm

import matplotlib.pyplot as plt



In [ ]:
label_dir = "/home/user/data_shared/T16TEK/BF_20250928T162949_20250928T162949_2.5m.tif"

with rasterio.open(label_dir) as src:
    img = src.read(1)

img

In [ ]:
class BuildingBinaryRaster(Dataset):
    """
    Dataset for co-registered binary building segmentation from mbf_binry.tif.
    - Assumes identical CRS/transform/shape to the Sentinel-2 tiles.
    - Uses the same px1/px2 exclusion and (160->128) window/crop as before.
    - Returns:
        s2data: (C,128,128) float in [0,1]
        label:  (128,128) float with values {0.0, 1.0}
    """
    def __init__(self,
                 top_dir,
                 s2_tiles,
                 labels,
                 training_bounds_left_top_right_bottom,
                 train_val_key,
                 complete_tile_size,
                 exclude_px1_px2,
                 patch_size=128,
                 buffer=32):

        self.top_dir = top_dir
        self.s2_tiles = s2_tiles
        self.labels = labels
        self.train_val_key = train_val_key
        self.patch_size = patch_size
        self.buffer = buffer

        locations = get_sample_locations(
            10980, 
            tb=training_bounds_left_top_right_bottom, 
            train_val_key=self.train_val_key,
            patch_size=patch_size,
            exclude_px1_px2=exclude_px1_px2
            )


        self.samples = []
        for loc in tqdm(locations, desc=f"Building {train_val_key} samples"):
            for sample_idx, time_str in enumerate(s2_tiles):
                # print(time_str)
                if "BF" not in time_str:
                    self.samples.append({
                        "sample_idx": int(sample_idx),
                        "time_str": time_str,
                        "location": loc
                    })
                else: continue

        np.random.shuffle(self.samples)
        print(f"{len(self.samples)} {train_val_key} samples after exclusion")

    def _get_dt_properties(self, time_str):

        capture_time = os.path.splitext(os.path.basename(time_str))[0]
        dt = datetime.strptime(capture_time, "%Y%m%dT%H%M%S")

        t0 = datetime(2015, 1, 1)
        delta = (dt - t0).total_seconds() / 86400.0  # days since t0

        # day-of-year
        doy = dt.timetuple().tm_yday  # 1..365/366
        doy_norm = (doy - 1) / 365.0
        doy_sin = math.sin(2 * math.pi * doy_norm)
        doy_cos = math.cos(2 * math.pi * doy_norm)

        return {"file_name": time_str,"delta_days": delta, "doy_sin": doy_sin, "doy_cos": doy_cos,}

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        s = self.samples[idx]
        dt_properties = self._get_dt_properties(s["time_str"])
        if self.train_val_key == "val":
            y_off, x_off = 16, 16
        else:
            y_off = np.random.randint(0, self.buffer + 1)
            x_off = np.random.randint(0, self.buffer + 1)

        s2_path = os.path.join(self.top_dir, self.s2_tiles[s["sample_idx"]])
        # ---- Read Sentinel-2 raster window & crop ----
        img = read_and_normalize_s2(
            s2_path,
            s,
            x_off,
            y_off,
            self.patch_size,
            win_size=160
        )

        # The labels are in 2.5 meter gsd, so need to scale the window by 4x
        # --- read binary mask window & crop (already aligned) ---
        with rasterio.open(os.path.join(self.top_dir, self.labels)) as src:
            win = Window(s["location"][1]*4, s["location"][0]*4, 160*4, 160*4)
            label = src.read(1, window=win)
            label = label[y_off*4:y_off*4 + self.patch_size*4, x_off*4:x_off*4 + self.patch_size*4]
        label = torch.from_numpy(label).long()  # (128,128) {0.0, 1.0}

        return {
            "timestamp": torch.tensor(dt_properties["delta_days"], dtype=torch.float32),
            "time_str": s["time_str"],
            "x_s2": x_off + s["location"][1],
            "y_s2": y_off + s["location"][0],
            "s2data": img,
            "label": label
        }


In [ ]:
def s2_to_rgb(s2_tensor, rgb_indices=None, eps=1e-6):
    """
    Convert (C,H,W) torch tensor in [0,1] to (H,W,3) numpy for plotting.
    rgb_indices: which channels correspond to R,G,B.
      Common cases:
        - If channels are [B02,B03,B04,...] => RGB is [B04,B03,B02] => indices [2,1,0]
        - If channels are [B04,B03,B02,...] => indices [0,1,2]
    """
    assert torch.is_tensor(s2_tensor), "s2_tensor must be torch.Tensor"
    assert s2_tensor.ndim == 3, f"Expected (C,H,W), got {tuple(s2_tensor.shape)}"

    C, H, W = s2_tensor.shape
    if rgb_indices is None:
        # Default guess: [B02,B03,B04,...] -> RGB = [B04,B03,B02]
        rgb_indices = [2, 1, 0] if C >= 3 else [0, 0, 0]

    rgb = s2_tensor[rgb_indices, :, :].detach().cpu().float().clamp(0, 1)
    rgb = rgb.permute(1, 2, 0).numpy()  # (H,W,3)

    # Optional contrast stretch for nicer plots (robust min/max)
    lo = np.quantile(rgb, 0.02)
    hi = np.quantile(rgb, 0.98)
    rgb = (rgb - lo) / (hi - lo + eps)
    rgb = np.clip(rgb, 0, 1)
    return rgb


def show_sample(sample, rgb_indices=None, alpha=0.35, title=None):
    """
    sample is one item returned by your dataset.
    """
    s2 = sample["s2data"]
    lab = sample["label"]

    if torch.is_tensor(lab):
        lab_np = lab.detach().cpu().numpy()
    else:
        lab_np = np.asarray(lab)

    rgb = s2_to_rgb(s2, rgb_indices=rgb_indices)

    fig, axs = plt.subplots(1, 3, figsize=(14, 4))
    if title is None:
        title = f'{sample.get("time_str","")}  |  x={sample.get("x_s2","?")} y={sample.get("y_s2","?")}'
    fig.suptitle(title)

    axs[0].imshow(rgb)
    axs[0].set_title("Sentinel-2 RGB")
    axs[0].axis("off")

    axs[1].imshow(lab_np, vmin=0, vmax=1)
    axs[1].set_title("Building label")
    axs[1].axis("off")

    axs[2].imshow(rgb)
    axs[2].imshow(lab_np, alpha=alpha, vmin=0, vmax=1)
    axs[2].set_title("Overlay")
    axs[2].axis("off")

    plt.tight_layout()
    plt.show()


In [ ]:
# ----------- YOU EDIT THESE -----------
top_dir = "/home/user/data_shared/T16TEK"

# list of S2 raster relative paths (as your class expects)
# e.g. ["T32UPU/20230101T100000.tif", ...] or whatever you currently pass in
s2_tiles = [
    '20221108T163511.tif', '20250814T162921.tif', '20251122T163621.tif', '20221009T163211.tif', '20220914T162839.tif', '20240918T162941.tif', '20240829T162901.tif', '20240211T163421.tif', '20231118T163549.tif', '20250518T163701.tif', '20250918T162839.tif', '20220107T163649.tif', '20240226T163139.tif', '20220721T162851.tif', '20230517T162831.tif', '20221029T163421.tif', '20221203T163649.tif', '20220507T162829.tif', '20250928T162949.tif', '20240406T162829.tif', '20230914T162911.tif', '20221004T163129.tif', '20240824T162829.tif', '20230412T162839.tif', '20220427T162829.tif', '20230924T163021.tif', '20241008T163201.tif', '20220621T162911.tif', '20250908T162829.tif', '20220313T163051.tif', '20250610T162829.tif', '20240725T162839.tif', '20221103T163439.tif', '20250915T163701.tif', '20231123T163611.tif', '20250225T163301.tif', '20241003T163029.tif', '20230711T162839.tif', '20231113T163531.tif', '20221014T163229.tif', '20241018T163311.tif', '20221019T163311.tif', '20230211T163419.tif']

# label filename relative to top_dir (single raster)
labels = "BF_20250928T162949_20250928T162949_2.5m.tif"

# training bounds in (left, top, right, bottom) in PIXEL coords of the 10m grid
training_bounds_left_top_right_bottom = (4000, 0, 5000, 5000)
complete_tile_size = 10980
px1 = (0, 4800) # We lack labels for this area, so we exclude it from training/validation
px2 = (3040, 6080)
# exclusion pixels (whatever you used before)
exclude_px1_px2 = (px1, px2)  # <-- replace with your real exclusion tuple/structure
# --------------------------------------


train_ds = BuildingBinaryRaster(
    top_dir=top_dir,
    s2_tiles=s2_tiles,
    labels=labels,
    training_bounds_left_top_right_bottom=training_bounds_left_top_right_bottom,
    train_val_key="train",
    complete_tile_size=complete_tile_size,
    exclude_px1_px2=exclude_px1_px2,
    patch_size=128,
    buffer=32,
)

val_ds = BuildingBinaryRaster(
    top_dir=top_dir,
    s2_tiles=s2_tiles,
    labels=labels,
    training_bounds_left_top_right_bottom=training_bounds_left_top_right_bottom,
    train_val_key="val",
    complete_tile_size=complete_tile_size,
    exclude_px1_px2=exclude_px1_px2,
    patch_size=128,
    buffer=32,
)

print("train len:", len(train_ds))
print("val len:", len(val_ds))


train_loader = DataLoader(
    train_ds,
    batch_size=8,
    shuffle=True,
    num_workers=4,
    pin_memory=True,
)

val_loader = DataLoader(
    val_ds,
    batch_size=8,
    shuffle=False,
    num_workers=4,
    pin_memory=True,
)


In [ ]:
sample = train_ds[0]

print("keys:", sample.keys())
print("time_str:", sample["time_str"])
print("timestamp:", sample["timestamp"], sample["timestamp"].dtype)
print("s2data:", tuple(sample["s2data"].shape), sample["s2data"].dtype,
      "min/max:", float(sample["s2data"].min()), float(sample["s2data"].max()))
print("label:", tuple(sample["label"].shape), sample["label"].dtype,
      "unique:", torch.unique(sample["label"]))

# Quick assertions (you can comment these out if needed)
assert sample["s2data"].ndim == 3 and sample["s2data"].shape[-1] == 128 and sample["s2data"].shape[-2] == 128
assert sample["label"].shape == (128*4, 128*4) or sample["label"].shape == (128, 128), \
    f"Label shape unexpected: {tuple(sample['label'].shape)}"


In [ ]:
train_ds

In [ ]:
# If your channels are [B02,B03,B04,...], keep [2,1,0]
# If your channels are [B04,B03,B02,...], use [0,1,2]
rgb_indices = [2, 1, 0]

for _ in range(4):
    i = random.randint(0, len(train_ds) - 1)
    s = train_ds[i]

    # If label is (512,512), plot it as-is (overlay still works, but shapes differ)
    # For overlay, we need same spatial shape -> downsample label for display if needed.
    lab = s["label"]
    if lab.shape[-1] != s["s2data"].shape[-1]:
        # display-only nearest downsample from 512->128
        lab_ds = torch.nn.functional.interpolate(
            lab[None, None].float(),
            size=(s["s2data"].shape[-2], s["s2data"].shape[-1]),
            mode="nearest"
        )[0, 0].long()
        s = dict(s)
        s["label"] = lab_ds

    show_sample(s, rgb_indices=rgb_indices, alpha=0.35)


In [ ]:
batch = next(iter(train_loader))
print("Batch keys:", batch.keys())
print("timestamp:", batch["timestamp"].shape, batch["timestamp"].dtype)
print("s2data:", batch["s2data"].shape, batch["s2data"].dtype)
print("label:", batch["label"].shape, batch["label"].dtype)
print("label unique (first batch):", torch.unique(batch["label"]))
